In [0]:
# Install required packages
%pip install mlflow>=2.9.0 databricks-sdk
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.9/611.9 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-8741014e-ff42-4383-8714-d180d7742fb2
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-8741014e-ff42-4383-8714-d180d7742fb2
    Can't

In [0]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from databricks import agents
import os

In [0]:
# Define a simple AI agent that can answer questions
class DataAnalysisAgent(mlflow.pyfunc.PythonModel):
    """
    A native Databricks AI Agent for data analysis tasks.
    This agent can answer questions about data and provide insights.
    """
    
    def __init__(self):
        self.name = "Data Analysis Agent"
        self.description = "An AI agent that helps with data analysis and insights"
    
    def predict(self, context, model_input):
        """
        Main prediction method for the agent.
        
        Args:
            context: MLflow context
            model_input: Dict with 'messages' key containing conversation history
        
        Returns:
            Dict with agent response
        """
        messages = model_input.get('messages', [])
        
        # Get the last user message
        if messages:
            last_message = messages[-1]
            user_query = last_message.get('content', '')
            
            # Simple response logic (you can enhance this with LLM calls)
            response = f"I'm your Data Analysis Agent. You asked: '{user_query}'. I'm ready to help with data analysis tasks!"
            
            return {
                "content": response,
                "role": "assistant"
            }
        
        return {
            "content": "Hello! I'm your Data Analysis Agent. How can I help you today?",
            "role": "assistant"
        }

# Create an instance of the agent
agent = DataAnalysisAgent()
print(f"✓ Agent '{agent.name}' created successfully!")

✓ Agent 'Data Analysis Agent' created successfully!


/local_disk0/.ephemeral_nfs/envs/pythonEnv-8741014e-ff42-4383-8714-d180d7742fb2/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [0]:
# Set up MLflow experiment
username = spark.sql("SELECT current_user()").collect()[0][0]
experiment_name = f"/Users/{username}/data_analysis_agent"
mlflow.set_experiment(experiment_name)

print(f"Using MLflow experiment: {experiment_name}")

# Log the agent model
with mlflow.start_run(run_name="data_analysis_agent_v1") as run:
    # Create input example
    input_example = {
        "messages": [
            {"role": "user", "content": "What insights can you provide?"}
        ]
    }
    
    # Infer signature
    signature = infer_signature(
        model_input=input_example,
        model_output={"content": "Sample response", "role": "assistant"}
    )
    
    # Log the model
    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent_model",
        python_model=agent,
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "mlflow>=2.9.0",
            "databricks-sdk"
        ]
    )
    
    print(f"\n✓ Agent model logged successfully!")
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
2026/08/19 09:13:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Using MLflow experiment: /Users/user@example.com/data_analysis_agent


🔗 View Logged Model at: https://your-workspace.cloud.databricks.com/ml/experiments/2545498113142240/models/m-48d0a564adc149d288fd95a53f6c7826?o=YOUR_WORKSPACE_ID
2026/08/19 09:13:24 INFO mlflow.pyfunc: Validating input example against model signature
2026/08/19 09:13:25 WARNING mlflow.models.model: Failed to validate serving input example {
  "messages": [
    {
      "role": "user",
    .... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.predict(model_uri=..., input_data=serving_input, env_manager="uv")` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().



✓ Agent model logged successfully!
Run ID: 43053d6500724377978a9bdc158b0701
Model URI: models:/m-48d0a564adc149d288fd95a53f6c7826


In [0]:
# Test the agent with a sample query
test_input = {
    "messages": [
        {"role": "user", "content": "Can you help me analyze sales data?"}
    ]
}

print("Testing the agent...\n")
response = agent.predict(context=None, model_input=test_input)
print(f"Agent Response:")
print(f"  Role: {response['role']}")
print(f"  Content: {response['content']}")
print(f"\n✓ Agent test completed successfully!")

Testing the agent...

Agent Response:
  Role: assistant
  Content: I'm your Data Analysis Agent. You asked: 'Can you help me analyze sales data?'. I'm ready to help with data analysis tasks!

✓ Agent test completed successfully!


In [0]:
# Register the agent to Unity Catalog for production use

# Define Unity Catalog model name
catalog_name = "sap_dev"  # Using your workspace catalog
schema_name = "default"  # Change to your schema
model_name = "data_analysis_agent"

uc_model_name = f"{catalog_name}.{schema_name}.{model_name}"

# Register to Unity Catalog
uc_model_info = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=uc_model_name
)

print(f"\n✓ Agent registered to Unity Catalog!")
print(f"Model Name: {uc_model_name}")
print(f"Version: {uc_model_info.version}")
print(f"\nYou can now:")
print(f"  1. View the model in the 'Agents' page")
print(f"  2. Deploy it as a model serving endpoint")
print(f"  3. Use it in production applications")

print("\n" + "="*60)
print("✓ AI Agent Setup Complete!")
print("="*60)

Successfully registered model 'sap_dev.default.data_analysis_agent'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]


✓ Agent registered to Unity Catalog!
Model Name: sap_dev.default.data_analysis_agent
Version: 1

You can now:
  1. View the model in the 'Agents' page
  2. Deploy it as a model serving endpoint
  3. Use it in production applications

✓ AI Agent Setup Complete!


🔗 Created version '1' of model 'sap_dev.default.data_analysis_agent': https://your-workspace.cloud.databricks.com/explore/data/models/sap_dev/default/data_analysis_agent/version/1?o=YOUR_WORKSPACE_ID


In [0]:
# Display final status and access links
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint = w.serving_endpoints.get(name="data-analysis-agent-endpoint")

print("=" * 80)
print("🎉 AI AGENT DEPLOYMENT COMPLETE")
print("=" * 80)

print("\n✅ SUCCESSFULLY DEPLOYED:")
print(f"   • Agent Model: sap_dev.default.data_analysis_agent (v1)")
print(f"   • Serving Endpoint: {endpoint.name}")
print(f"   • Status: {endpoint.state.config_update if endpoint.state else 'Unknown'}")

print("\n📍 QUICK ACCESS:")
print(f"   • View in Agents page: AI/ML > Agents")
print(f"   • Monitor Serving: Serving > {endpoint.name}")
print(f"   • Model Details: Unity Catalog > sap_dev > default > data_analysis_agent")

print("\n🔧 NEXT STEPS:")
print("   1. Wait 5-10 minutes for endpoint to become READY")
print("   2. Test the agent via the Serving endpoint")
print("   3. Integrate into your applications via API")
print("   4. Monitor usage and performance in the Serving dashboard")

print("\n" + "=" * 80)

🎉 AI AGENT DEPLOYMENT COMPLETE

✅ SUCCESSFULLY DEPLOYED:
   • Agent Model: sap_dev.default.data_analysis_agent (v1)
   • Serving Endpoint: data-analysis-agent-endpoint
   • Status: EndpointStateConfigUpdate.IN_PROGRESS

📍 QUICK ACCESS:
   • View in Agents page: AI/ML > Agents
   • Monitor Serving: Serving > data-analysis-agent-endpoint
   • Model Details: Unity Catalog > sap_dev > default > data_analysis_agent

🔧 NEXT STEPS:
   1. Wait 5-10 minutes for endpoint to become READY
   2. Test the agent via the Serving endpoint
   3. Integrate into your applications via API
   4. Monitor usage and performance in the Serving dashboard

